# StageNet + LRP (Layer-wise Relevance Propagation)

This notebook demonstrates using **Layer-wise Relevance Propagation (LRP)** with **StageNet** for mortality prediction on **MIMIC-III** (mounted from Campus Cluster).

## What is LRP?

LRP is an explainability method that decomposes a neural network's prediction into relevance scores for each input feature:

**Key Properties:**
- ✓ Single backward pass (fast!)
- ✓ No baseline required
- ✓ Conservation: relevances sum to f(x)
- ✓ Multiple propagation rules available

**LRP Rules:**
1. **Epsilon rule (ε-rule)**: Numerically stable, smoother attributions
2. **AlphaBeta rule (αβ-rule)**: Sharper visualizations, highlights positive evidence

## Comparison with Other Methods

| Method | Speed | Baseline? | Sum Property |
|--------|-------|-----------|-------------|
| LRP | Fast (1 pass) | No | ∑ = f(x) |
| Integrated Gradients | Slow (many passes) | Yes | ∑ = f(x) - f(baseline) |
| DeepLift | Fast (1 pass) | Yes | ∑ = f(x) - f(baseline) |
| SHAP | Variable | Yes | Local accuracy |

## Setup

In [11]:
from pathlib import Path

import torch
import numpy as np

from pyhealth.datasets import (
    MIMIC3Dataset,
    get_dataloader,
    split_by_patient,
)
from pyhealth.interpret.methods import LayerwiseRelevancePropagation
from pyhealth.models import StageNet
from pyhealth.tasks import MortalityPredictionMIMIC3
from pyhealth.trainer import Trainer

## Load MIMIC-III Dataset (Mounted from Campus Cluster)

## Data Access

**Full MIMIC-IV dataset** requires PhysioNet credentialing.
See the [Getting MIMIC Access Guide](https://docs.google.com/document/d/1NHgXzSPINafSg8Cd_whdfSauFXgh-ZflZIw5lu6k2T0/edit?usp=sharing) for instructions.

**Free 100-patient demo** (no credentialing required):
```
wget -r -N -c -np --user <physionet_user> --ask-password \
    https://physionet.org/files/mimic-iv-demo/2.2/
```
Or download via the PhysioNet web UI at: https://physionet.org/content/mimic-iv-demo/2.2/

### Required folder layout

`MIMIC4_EHR_ROOT` must be the directory that contains both `hosp/` and `icu/` subdirectories:

```
<MIMIC4_EHR_ROOT>/
├── hosp/
│   ├── patients.csv.gz        ← patients table
│   ├── admissions.csv.gz      ← admissions table
│   ├── diagnoses_icd.csv.gz   ← diagnoses_icd table
│   ├── procedures_icd.csv.gz  ← procedures_icd table
│   ├── labevents.csv.gz       ← labevents table
│   └── d_labitems.csv.gz      ← joined automatically by labevents
└── icu/
    └── icustays.csv.gz        ← always loaded (default table)
```

Set `MIMIC4_EHR_ROOT` in the cell below, then run the preflight check.

In [12]:
from pathlib import Path

# ── USING MIMIC-III FROM CAMPUS CLUSTER ──────────────────────────────────
# Mounted via sshfs from campus-cluster:/projects/illinois/eng/cs/jimeng/mimic3
# To mount: sshfs campus-cluster:/projects/illinois/eng/cs/jimeng/mimic3 ~/mimic-data
# To unmount: fusermount -u ~/mimic-data
# ──────────────────────────────────────────────────────────────────────────
MIMIC3_ROOT = str(Path.home() / "mimic-data")

# Tables required for MIMIC-III mortality prediction
REQUIRED_TABLES = [
    "DIAGNOSES_ICD",
    "PROCEDURES_ICD",
    "PRESCRIPTIONS",
]

# Preflight: verify required files exist
root = Path(MIMIC3_ROOT)
table_files = {
    "DIAGNOSES_ICD":  root / "DIAGNOSES_ICD.csv",
    "PROCEDURES_ICD": root / "PROCEDURES_ICD.csv",
    "PRESCRIPTIONS":  root / "PRESCRIPTIONS.csv",
    "ADMISSIONS":     root / "ADMISSIONS.csv",
    "PATIENTS":       root / "PATIENTS.csv",
}
missing = [str(p) for p in table_files.values() if not p.exists()]
if missing:
    print("⚠️  Missing files — check if mimic-data is mounted:")
    for f in missing:
        print(f"   {f}")
    print("\nTo mount: sshfs campus-cluster:/projects/illinois/eng/cs/jimeng/mimic3 ~/mimic-data")
else:
    print(f"✓ All required files found under: {MIMIC3_ROOT}")

✓ All required files found under: /home/nemine/mimic-data


In [13]:
from pyhealth.datasets import MIMIC3Dataset

base_dataset = MIMIC3Dataset(
    root=MIMIC3_ROOT,
    tables=REQUIRED_TABLES,
    dev=True,  # Use dev mode for faster testing (1000 patients)
)

print(f"Loaded {len(base_dataset.unique_patient_ids)} patients")

No config path provided, using default config
Initializing mimic3 dataset from /home/nemine/mimic-data (dev mode: True)
No cache_dir provided. Using default cache dir: /home/nemine/.cache/pyhealth/dd96d4ce-639a-5750-851a-19cbdba4da7c
Found 1000 unique patient IDs
Loaded 1000 patients


## Apply Task and Create Sample Dataset

In [14]:
cache_dir = Path("../../mimic3_stagenet_lrp_cache_nb")

print("Applying mortality prediction task...")
sample_dataset = base_dataset.set_task(
    MortalityPredictionMIMIC3(),
    cache_dir=str(cache_dir),
)

print(f"Total samples: {len(sample_dataset)}")

Applying mortality prediction task...
Setting task MortalityPredictionMIMIC3 for mimic3 base dataset...
Found cached processed samples at ../../mimic3_stagenet_lrp_cache_nb/samples_8247e486-59e0-5fff-be52-2529fef9a812.ld, skipping processing.
Total samples: 208


## Split Dataset and Create DataLoaders

In [15]:
train_dataset, val_dataset, test_dataset = split_by_patient(
    sample_dataset, [0.8, 0.1, 0.1]
)

train_loader = get_dataloader(train_dataset, batch_size=64, shuffle=True)
val_loader = get_dataloader(val_dataset, batch_size=64, shuffle=False)
test_loader = get_dataloader(test_dataset, batch_size=1, shuffle=False)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Train: 156 | Val: 16 | Test: 36


## Initialize and Train StageNet Model

In [16]:
model = StageNet(
    dataset=sample_dataset,
    embedding_dim=128,
    chunk_size=128,
    levels=3,
    dropout=0.3,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 7,183,625


In [17]:
trainer = Trainer(
    model=model,
    device="cpu",  # Change to "cuda" if GPU available
    metrics=["pr_auc", "roc_auc", "accuracy", "f1"],
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=5,
    monitor="roc_auc",
    optimizer_params={"lr": 1e-4},
)

StageNet(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(712, 128, padding_idx=0)
    (procedures): Embedding(258, 128, padding_idx=0)
    (drugs): Embedding(763, 128, padding_idx=0)
  ))
  (stagenet): ModuleDict(
    (conditions): StageNetLayer(
      (kernel): Linear(in_features=129, out_features=1542, bias=True)
      (recurrent_kernel): Linear(in_features=385, out_features=1542, bias=True)
      (nn_scale): Linear(in_features=384, out_features=64, bias=True)
      (nn_rescale): Linear(in_features=64, out_features=384, bias=True)
      (nn_conv): Conv1d(384, 384, kernel_size=(10,), stride=(1,))
      (nn_dropconnect): Dropout(p=0.3, inplace=False)
      (nn_dropconnect_r): Dropout(p=0.3, inplace=False)
      (nn_dropout): Dropout(p=0.3, inplace=False)
      (nn_dropres): Dropout(p=0.3, inplace=False)
    )
    (procedures): StageNetLayer(
      (kernel): Linear(in_features=129, out_features=1542, bias=True)
      (recurrent_kernel): Line

Epoch 0 / 5:   0%|          | 0/3 [00:00<?, ?it/s]

/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'conditions' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'procedures' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'drugs' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(


--- Train epoch-0, step-3 ---
loss: 0.6889


Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'conditions' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'procedures' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'drugs' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
Evaluation: 10

...

Epoch 4 / 5:   0%|          | 0/3 [00:00<?, ?it/s]

/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'conditions' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'procedures' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'drugs' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(


--- Train epoch-4, step-15 ---
loss: 0.6136


Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'conditions' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'procedures' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'drugs' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
Evaluation: 10

--- Eval epoch-4, step-15 ---
pr_auc: 0.3269
roc_auc: 0.5714
accuracy: 0.8750
f1: 0.0000
loss: 0.5936
Loaded best model


## Evaluate Model

In [18]:
results = trainer.evaluate(test_loader)
print("\nTest Results:")
for metric, value in results.items():
    print(f"  {metric}: {value:.4f}")

Evaluation:   0%|          | 0/36 [00:00<?, ?it/s]/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'conditions' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'procedures' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'drugs' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
Evaluation: 1


Test Results:
  pr_auc: 0.5714
  roc_auc: 0.8828
  accuracy: 0.9167
  f1: 0.4000
  loss: 0.6004


## Get Test Sample for Interpretation

In [21]:
sample_batch = next(iter(test_loader))

# Get model prediction
with torch.no_grad():
    output = model(**sample_batch)
    probs = output["y_prob"]
    preds = torch.argmax(probs, dim=-1)
    true_label = sample_batch[model.label_key]

print("Model Prediction:")
print(f"  True label: {int(true_label[0].item())}")
print(f"  Predicted class: {int(preds[0].item())}")
print(f"  P(Survived): {probs[0, 0].item():.4f}")
if probs.shape[-1] > 1:
    print(f"  P(Died): {probs[0, 1].item():.4f}")

/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'conditions' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'procedures' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:559: UserWarning: Feature 'drugs' is not a temporal tuple. Using fallback mode without time intervals. The model may not learn temporal patterns properly. Please use 'stagenet' or 'stagenet_tensor' processors in your input schema.
  warnings.warn(


Model Prediction:
  True label: 0
  Predicted class: 0
  P(Survived): 0.4164


## Apply LRP with Epsilon Rule

In [22]:
# Initialize LRP with epsilon rule (numerically stable)
lrp_epsilon = LayerwiseRelevancePropagation(
    model, 
    rule="epsilon", 
    epsilon=0.01,
    use_embeddings=True
)

# Compute attributions
attributions_epsilon = lrp_epsilon.attribute(**sample_batch)

print("\nLRP Epsilon-Rule Attributions:")
for feature_key, attr in attributions_epsilon.items():
    total_relevance = attr[0].sum().item()
    positive = attr[0][attr[0] > 0].sum().item()
    negative = attr[0][attr[0] < 0].sum().item()
    
    print(f"\n{feature_key}:")
    print(f"  Shape: {attr.shape}")
    print(f"  Total relevance: {total_relevance:+.6f}")
    print(f"  Positive: {positive:+.6f} | Negative: {negative:+.6f}")


LRP Epsilon-Rule Attributions:

conditions:
  Shape: torch.Size([1])
  Total relevance: +0.000003
  Positive: +0.000003 | Negative: +0.000000

procedures:
  Shape: torch.Size([1])
  Total relevance: +0.000003
  Positive: +0.000003 | Negative: +0.000000

drugs:
  Shape: torch.Size([1])
  Total relevance: +0.000003
  Positive: +0.000003 | Negative: +0.000000


## Apply LRP with AlphaBeta Rule

In [23]:
# Initialize LRP with alphabeta rule (sharper visualizations)
lrp_alphabeta = LayerwiseRelevancePropagation(
    model,
    rule="alphabeta",
    alpha=1.0,
    beta=0.0,
    use_embeddings=True
)

# Compute attributions
attributions_alphabeta = lrp_alphabeta.attribute(**sample_batch)

print("\nLRP AlphaBeta-Rule Attributions:")
for feature_key, attr in attributions_alphabeta.items():
    total_relevance = attr[0].sum().item()
    positive = attr[0][attr[0] > 0].sum().item()
    negative = attr[0][attr[0] < 0].sum().item()
    
    print(f"\n{feature_key}:")
    print(f"  Shape: {attr.shape}")
    print(f"  Total relevance: {total_relevance:+.6f}")
    print(f"  Positive: {positive:+.6f} | Negative: {negative:+.6f}")


LRP AlphaBeta-Rule Attributions:

conditions:
  Shape: torch.Size([1])
  Total relevance: -0.005460
  Positive: +0.000000 | Negative: -0.005460

procedures:
  Shape: torch.Size([1])
  Total relevance: -0.005460
  Positive: +0.000000 | Negative: -0.005460

drugs:
  Shape: torch.Size([1])
  Total relevance: -0.005460
  Positive: +0.000000 | Negative: -0.005460


## Visualize Top Contributing Features

In [ ]:
def show_top_features(attributions, feature_key, top_k=10):
    """Show top contributing features."""
    if feature_key not in attributions:
        print(f"\nFeature '{feature_key}' not found in attributions")
        print(f"Available features: {list(attributions.keys())}")
        return
        
    attr = attributions[feature_key][0]
    flat_attr = attr.flatten()
    
    # Get top k by absolute value
    k = min(top_k, flat_attr.numel())
    top_vals, top_indices = torch.topk(flat_attr.abs(), k=k)
    
    print(f"\nTop {k} contributing features for '{feature_key}':")
    
    # Generic display for all features
    for i, idx in enumerate(top_indices.tolist(), 1):
        relevance = flat_attr[idx].item()
        print(f"  {i:2d}. Index={idx:4d} → Relevance={relevance:+.6f}")

# Show available features
print("Available features in attributions:", list(attributions_epsilon.keys()))
print()

# Show top features for both rules
print("="*70)
print("EPSILON RULE - Top Contributing Features")
print("="*70)
for feature in attributions_epsilon.keys():
    show_top_features(attributions_epsilon, feature, top_k=10)

print("\n" + "="*70)
print("ALPHABETA RULE - Top Contributing Features")
print("="*70)
for feature in attributions_alphabeta.keys():
    show_top_features(attributions_alphabeta, feature, top_k=10)

EPSILON RULE - Top Contributing Features


KeyError: 'labs'

## Summary and Key Takeaways

### When to Use LRP:

✅ **Use LRP when:**
- You need fast attributions (single backward pass)
- Conservation property is important (sum to f(x))
- No obvious baseline exists
- Want to try different propagation rules
- Debugging/understanding model decisions

❌ **Consider alternatives when:**
- Need theoretical guarantees → Use **Integrated Gradients**
- Have a good baseline and ReLU-heavy model → Use **DeepLift**
- Need game-theoretic explanation → Use **SHAP**
- Want local linear approximation → Use **LIME**

### LRP Rules:

1. **Epsilon Rule (ε=0.01)**
   - More stable
   - Smoother attributions
   - Good for general interpretation

2. **AlphaBeta Rule (α=1.0, β=0.0)**
   - Sharper visualizations
   - Highlights positive evidence
   - Better for identifying key features

### Conservation Property:

LRP satisfies: **Σ(relevances) ≈ f(x)**

This means the sum of all input relevances should approximately equal the model's output for the target class.

### Next Steps:

- Try different epsilon values (0.001, 0.1, 1.0)
- Experiment with alphabeta ratios
- Compare with other methods (IG, DeepLift, SHAP)
- Apply to different prediction tasks
- Use for model debugging and validation